# Week 1 — live-coding notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week01/week1_demos.ipynb)

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Use these links during the lecture breaks; you do not need to scroll through or rerun the entire notebook.

### Deck A

- [Demo 1 — A neural network learns an option-pricing curve](#demo-1-a-neural-network-learns-an-option-pricing-curve)
- [Demo 2 — Audit four strategies across a frozen boundary](#demo-2-audit-four-strategies-across-a-frozen-boundary)

### Deck B

- [Demo 3 — A random walk and its returns side by side](#demo-3-a-random-walk-and-its-returns-side-by-side)
- [Real-data companion — US market wealth, returns, and tail days](#real-data-companion-us-market-wealth-returns-and-tail-days)
- [Demo 4 — Read memory from the ACF](#demo-4-read-memory-from-the-acf)
- [Demo 5 — Spurious regression on two independent random walks](#demo-5-spurious-regression-on-two-independent-random-walks)
- [Demo 6 — A cointegrated pair, and its stationary spread](#demo-6-a-cointegrated-pair-and-its-stationary-spread)
- [Demo 7 — Multiple-testing teaser (connects to Part A and Ch 16)](#demo-7-multiple-testing-teaser-connects-to-part-a-and-ch-16)

## Your Week 1 practice research record

Complete each entry in the editable Markdown cell immediately below its demo. **Do not write in this overview.** Those local entries, taken together, are your Week 1 practice record. After writing each entry, open the collapsed completed example and compare its specificity with yours.

These entries are practice. Keep them in this weekly notebook; Weeks 1–5 practice records are not merged or submitted separately. When your project begins in Week 6, you will start one project-root `RESEARCH_RECORD.md` and maintain that file through the final submission.

### What you will record this week

- **Demo 1 — Learning brief:** define the target, why it may be learnable, what evidence would count, and what decision could change.
- **Demo 2 — Four-strategy audit:** compare the development and later rankings, then state the narrow conclusion.
- **Demo 3 — Transformation:** name the object, units, transformation, sample, and what the transformation did not fix.
- **Optional real-data companion:** compare the controlled simulation with historical market returns without changing the procedure.
- **Demo 4 — Memory:** describe the ACF pattern and what it cannot establish.
- **Demos 5–6 — Levels and spreads:** separate a misleading level regression from a cointegrated relation.
- **Demo 7 — Search:** record how many candidates were compared and why the winner is not evidence of skill.

For every result, include a limitation and only the narrow claim the evidence supports.

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
from io import BytesIO, StringIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.stattools import acf, adfuller, coint

def course_csv(relative_path):
    """Read bundled Fama-French data locally, or its public source in Colab."""
    for root in (Path.cwd(), *Path.cwd().parents):
        local = root / relative_path
        if local.exists():
            return pd.read_csv(local), str(local)
    archives = {
        "datasets/famafrench/ff_factors_daily.csv": "F-F_Research_Data_Factors_daily_CSV.zip",
        "datasets/famafrench/ff_12industry_daily.csv": "12_Industry_Portfolios_daily_CSV.zip",
    }
    archive = archives[relative_path]
    url = f"https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{archive}"
    with urlopen(url) as response, ZipFile(BytesIO(response.read())) as zipped:
        lines = zipped.read(zipped.namelist()[0]).decode("utf-8").splitlines()
    header_row = next(i for i, line in enumerate(lines) if line.startswith(","))
    rows = []
    for line in lines[header_row + 1:]:
        first = line.split(",", 1)[0].strip()
        if len(first) == 8 and first.isdigit():
            rows.append(line)
        elif rows:
            break
    frame = pd.read_csv(StringIO("\n".join([lines[header_row], *rows])))
    return frame.rename(columns={frame.columns[0]: "date"}), url


def load_market_returns():
    """Return daily US market simple returns and their source."""
    market_raw, market_source = course_csv(
        "datasets/famafrench/ff_factors_daily.csv"
    )
    market_raw["date"] = pd.to_datetime(
        market_raw["date"].astype(str), format="%Y%m%d"
    )
    market_data = market_raw.set_index("date").sort_index()
    market_daily_return = (market_data["Mkt-RF"] + market_data["RF"]) / 100.0
    return market_daily_return, market_source


def adf_summary(name, x, regression="c", autolag="AIC"):
    """Print and return a compact description of one ADF test."""
    stat, p_value, selected_lag, nobs, *_ = adfuller(
        x, regression=regression, autolag=autolag
    )
    conclusion = (
        "reject the specified unit-root null"
        if p_value < 0.05
        else "fail to reject the specified unit-root null"
    )
    print(
        f"{name}: N={len(x)}, regression='{regression}', autolag='{autolag}', "
        f"selected lag={selected_lag}, ADF statistic={stat:+.3f}, "
        f"p-value={p_value:.4f} -> {conclusion}"
    )
    return {
        "series": name,
        "n": len(x),
        "regression": regression,
        "autolag": autolag,
        "selected_lag": selected_lag,
        "nobs_used": nobs,
        "statistic": stat,
        "p_value": p_value,
        "conclusion": conclusion,
    }

<a id="demo-1-a-neural-network-learns-an-option-pricing-curve"></a>

### Demo 1 — A neural network learns an option-pricing curve · Deck A · run at the break after S1

> **First demonstration:** Deck A, after recording segment S1. Run the cell exactly as written to reproduce the option-pricing figure. You are not expected to understand every line yet; read the graph, then continue to the second pause and editable learning-brief cell immediately below the output.

In [ ]:
# Just run this first example as written. We will unpack the machinery later.
def black_scholes_call(moneyness, maturity=1.0, volatility=0.2, rate=0.0):
    """Black–Scholes call price divided by strike, as a function of S/K."""
    m = np.asarray(moneyness)
    d1 = (np.log(m) + (rate + 0.5 * volatility**2) * maturity) / (
        volatility * np.sqrt(maturity)
    )
    d2 = d1 - volatility * np.sqrt(maturity)
    return m * norm.cdf(d1) - np.exp(-rate * maturity) * norm.cdf(d2)


option_rng = np.random.default_rng(1994)
train_x = np.sort(option_rng.uniform(0.68, 1.32, 220))
train_y = black_scholes_call(train_x)

# The network receives examples, not the Black–Scholes formula.
option_network = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(32, 32),
        activation="tanh",
        solver="lbfgs",
        alpha=1e-7,
        max_iter=5000,
        random_state=1994,
    ),
)
option_network.fit(train_x[:, None], train_y)

grid = np.linspace(0.55, 1.45, 500)
truth = black_scholes_call(grid)
learned = option_network.predict(grid[:, None])
inside = (grid >= 0.68) & (grid <= 1.32)
abs_error = np.abs(learned - truth) * 10_000

fig, (curve, error) = plt.subplots(
    1, 2, figsize=(14, 4.7),
    gridspec_kw={"width_ratios": [3.4, 1.25], "wspace": 0.28},
)
curve.axvspan(0.68, 1.32, color="tab:blue", alpha=0.07,
              label="training domain")
curve.scatter(train_x[::5], train_y[::5], s=22, color="tab:orange",
              alpha=0.65, label="examples", zorder=2)
curve.plot(grid, truth, color="black", ls="--", lw=2.3,
           label="Black–Scholes")
curve.plot(grid, learned, color="tab:green", lw=2.7,
           label="neural network")
curve.set(xlabel="moneyness  $S/K$", ylabel="normalized call price  $C/K$",
          title="Pricing function")
curve.legend(frameon=False, ncol=2, loc="upper left")
curve.spines[["top", "right"]].set_visible(False)

error.axvspan(0.68, 1.32, color="tab:blue", alpha=0.07)
error.plot(grid, abs_error, color="tab:red", lw=2)
error.fill_between(grid[inside], 0, abs_error[inside],
                   color="tab:red", alpha=0.18)
error.set(xlabel="moneyness  $S/K$", ylabel="basis points of strike",
          title="Absolute error")
error.spines[["top", "right"]].set_visible(False)

fig.suptitle("Examples teach a network the Black–Scholes pricing curve",
             fontweight="bold", fontsize=17)
plt.show()

---

### Pause again: turn the demonstration into a research plan

Demo 1 and the learning brief have two different purposes. The code above lets
you see a neural network learn a nonlinear function from examples. The short
exercise below asks you to practice a separate research habit: state what a
fair learning experiment would need **before** you fit it. In the graph,
*moneyness* is the stock price divided by the strike price; a value near 1 means
the option is approximately at the money. One basis point is 0.01 percent of
the strike price.

Choose option pricing, insurance claim frequency, or daily return direction
only as a practice problem. **You are not choosing or committing to your final
project here.** Weeks 1–5 keep these practice entries in the weekly notebooks.
Your final-project `RESEARCH_RECORD.md` begins in Week 6, after the project topic
is chosen. You may reuse this idea later if it genuinely fits, but you do not
have to.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| target | `[outcome and units]` |
| source of structure | `[why it may be learnable]` |
| evidence | `[approach, baseline, later observations, and score]` |
| decision | `[specific action that would change]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

This completed example deliberately uses a different allowed topic. Compare the
structure of its four answers with yours rather than copying the demonstration.

**Completed learning brief — insurance claim frequency**

1. **Target:** I want to estimate the number of claims per policy-year, measured as a nonnegative count.
2. **Source of structure:** Claim frequency may be learnable because driver, vehicle, region, and history can interact in stable but nonlinear ways.
3. **Evidence:** On later policies, I would compare gradient boosting with a constant-rate forecast and a Poisson regression using Poisson deviance, a score designed for count predictions.
4. **Decision:** If the richer model lowers later deviance and remains stable across subgroups, the insurer could change pricing and reserve estimates.

This is a plan, not a result. It names what would count as evidence before the
contest is run. Your topic may differ; it still needs these four sentences.

</details>

<a id="demo-2-audit-four-strategies-across-a-frozen-boundary"></a>

### Demo 2 — Audit four strategies across a frozen boundary · Deck A · run at the break after S2

> **Audit demonstration:** Deck A, after recording segment S2. Before running the cell, rank A–D using only the development-period chart in the lecture. Then run the cell to reveal the procedures and later-period evidence, and complete the four-strategy audit in the editable Markdown cell below the output.

In [ ]:
# Before running: rank A–D using only the development-period chart in the lecture.
# These annual observations are a compact, self-contained extract from the
# chapter experiment. Students do not need the chapter notebook or its assets.
audit_csv = """Date,A development,B development,C development,D development,equal-risk development,60/40 development,A later,B later,C later,D later,equal-risk later,60/40 later
1990-12-31,0.096331,0.166288,0.025916,0.040344,0.026749,-0.018902,,,,,,
1991-12-31,0.270206,0.398369,0.221376,0.223311,0.222209,0.185371,,,,,,
1992-12-31,0.310507,0.542604,0.272951,0.274886,0.273784,0.231518,,,,,,
1993-12-31,0.386650,0.693849,0.376993,0.378927,0.377826,0.325896,,,,,,
1994-12-31,0.431311,0.813902,0.322752,0.307956,0.323585,0.275938,,,,,,
1995-12-31,0.590074,1.066986,0.586694,0.519213,0.587527,0.547363,,,,,,
1996-12-31,0.631774,1.237940,0.650335,0.558830,0.653472,0.657060,,,,,,
1997-12-31,0.723392,1.524769,0.804550,0.702966,0.806298,0.867731,,,,,,
1998-12-31,0.919045,1.792613,0.981374,0.877850,0.983122,1.065427,,,,,,
1999-12-31,0.993572,1.904541,0.965881,0.862281,0.967629,1.164899,,,,,,
2000-12-31,1.192106,2.100887,1.063914,0.926406,1.065662,1.134903,,,,,,
2001-12-31,1.331326,2.378850,1.063601,0.960968,1.060542,1.060145,,,,,,
2002-12-31,1.548368,2.693418,1.100121,1.041252,1.094866,0.957544,,,,,,
2003-12-31,1.678232,2.992079,1.171960,1.069607,1.171483,1.092987,,,,,,
2004-12-31,1.729482,3.045472,1.222027,1.106493,1.222508,1.162173,,,,,,
2005-12-31,1.747063,3.064063,1.245984,1.117567,1.247595,1.193522,,,,,,
2006-12-31,1.766374,3.111119,1.299186,1.138477,1.300797,1.281186,,,,,,
2007-12-31,1.905863,3.164838,1.372335,1.211626,1.373946,1.338262,,,,,,
2008-12-31,2.239120,3.600839,1.408529,1.340622,1.409775,1.151128,,,,,,
2009-12-31,2.501040,4.159480,1.398912,1.303651,1.415757,1.209716,,,,,,
2010-12-31,2.620048,4.353163,1.494437,1.368384,1.511281,1.306572,,,,,,
2011-12-31,2.778311,4.659325,1.593648,1.449243,1.610493,1.368317,,,,,,
2012-12-31,2.855879,4.699517,1.636808,1.488756,1.652203,1.446510,,,,,,
2013-12-31,2.876811,4.753885,1.651963,1.554208,1.674930,1.566648,,,,,,
2014-12-31,2.901256,4.782360,1.740670,1.605164,1.763637,1.667241,,,,,,
2015-12-31,2.986344,4.826528,1.730397,1.583812,1.757009,1.667174,,,,,,
2016-12-31,,,,,,,-0.071805,-0.008148,0.017358,-0.007848,0.025960,0.057876
2017-12-31,,,,,,,-0.094435,0.015968,0.099162,0.059565,0.107764,0.180430
2018-12-31,,,,,,,-0.190159,0.011526,0.102145,0.048754,0.109916,0.141186
2019-12-31,,,,,,,-0.223667,0.060602,0.213997,0.146344,0.225395,0.334958
2020-12-31,,,,,,,-0.303455,0.035772,0.278480,0.185249,0.299239,0.464004
2021-12-31,,,,,,,-0.369369,0.026421,0.314014,0.214940,0.334773,0.627149
2022-12-31,,,,,,,-0.597438,-0.095679,0.133315,0.145143,0.167337,0.427354
2023-12-31,,,,,,,-0.630700,-0.080795,0.241021,0.189581,0.276502,0.599762
2024-12-31,,,,,,,-0.707835,-0.110017,0.306231,0.218679,0.341712,0.762172
2025-12-31,,,,,,,-0.814853,-0.136956,0.375093,0.274179,0.421664,0.898920
2026-12-31,,,,,,,-0.840808,-0.174212,0.390377,0.295696,0.443181,0.973976
"""
audit = pd.read_csv(StringIO(audit_csv), parse_dates=["Date"]).set_index("Date")

colors = {"A": "tab:red", "B": "tab:orange", "C": "tab:blue", "D": "tab:green"}
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
for key, color in colors.items():
    axes[0].plot(audit.index, audit[f"{key} development"], label=key, color=color, lw=2)
    axes[1].plot(audit.index, audit[f"{key} later"], label=key, color=color, lw=2)
for ax, suffix in zip(axes, ("development", "later")):
    ax.plot(audit.index, audit[f"equal-risk {suffix}"], color="0.35", ls="--", label="equal-risk")
    ax.plot(audit.index, audit[f"60/40 {suffix}"], color="0.7", ls=":", label="60/40")
    ax.axhline(0, color="black", lw=0.6)
    ax.set_ylabel("cumulative log return")
    ax.legend(frameon=False, ncol=2, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_title("1. Development period: the tempting ranking")
axes[1].set_title("2. Frozen later period: the ranking changes")
fig.tight_layout()
plt.show()

audit_key = pd.DataFrame(
    {
        "model": [
            "degree-10 polynomial OLS",
            "degree-10 polynomial OLS",
            "Lasso",
            "fixed 12-month momentum rule",
        ],
        "features": [
            "four noise series",
            "four real momentum windows",
            "four real momentum windows",
            "one 12-month momentum signal",
        ],
        "selection / validation": [
            "none",
            "none",
            "walk-forward; expanding window",
            "rule fixed in advance",
        ],
    },
    index=["A", "B", "C", "D"],
)
print(audit_key.to_string())
print("\nStored values are year-end extracts from the course's full daily backtest.")
print("The later period can reject a procedure here; it cannot establish a universal winner.")

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| development ranking | `[A–D and your ranking rule]` |
| later ranking | `[A–D after the frozen boundary]` |
| evidence that changed the decision | `[specific result or procedure]` |
| further test before funding | `[additional evidence]` |
| narrow claim | `[what this period warrants and what it does not]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed four-strategy audit**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| development ranking | B, A, then C and D approximately tied; rule: development Sharpe and cumulative growth |
| later ranking | C, D, B, A by the end of the frozen later period; C and D remain positive while A and B lose money |
| evidence that changed the decision | A used noise and B used real features, but both fitted roughly 1,000 polynomial terms to the displayed development history; neither strategy's performance persisted after 2015 |
| further test before funding | repeat the frozen procedure across other assets, starting dates, and cost assumptions without retuning it |
| narrow claim | in this one later period, the simpler or more strongly constrained procedures held up better; this does not establish that C or D is universally superior |

The later panel rejects the strongest development story. It does not reveal the
strategy that must win next.

</details>

<a id="demo-3-a-random-walk-and-its-returns-side-by-side"></a>

### Demo 3 — A random walk and its returns side by side · Deck B · run at the break after S4

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
T = 1000
demo3_rng = np.random.default_rng(0)
steps = demo3_rng.standard_normal(T) * 0.01
prices = 100 * np.exp(np.cumsum(steps))
simple_returns = np.diff(prices) / prices[:-1]
returns = np.diff(np.log(prices))

print(f"Maximum |simple - log return|: {np.max(np.abs(simple_returns - returns)):.2e}")
print(f"Cumulative simple return: {prices[-1] / prices[0] - 1:+.2%}")
print(f"Sum of log returns:       {returns.sum():+.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(prices, lw=0.8)
axes[0].set_title("Price (non-stationary — drifts without bound)")
axes[0].set_xlabel("t"); axes[0].set_ylabel("Price")
axes[1].plot(returns, lw=0.5, color="tab:red")
axes[1].plot(simple_returns, lw=0.4, color="tab:blue", alpha=0.6,
             label="simple return")
axes[1].set_title("Log return (stationary — bounded scale)")
axes[1].set_xlabel("t"); axes[1].set_ylabel("Log return")
axes[1].legend()
fig.tight_layout()
plt.show()
# Expected: the two conventions are close on daily moves, yet the totals differ:
#           log returns sum to about -0.482 while simple returns compound to about -38.2%.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| object and units | `[series and measurement unit]` |
| transformation | `[operation applied]` |
| sample span | `[observations, dates if available, and seed]` |
| what it did **not** fix | `[remaining problem or instability]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**What a complete entry looks like** — the slide asks for *the object and its
units; the transformation applied; sample span; what the transformation did not
fix*. Yours will differ; the level of detail should not.

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| object and units | simulated daily close, index points (arbitrary base 100) |
| transformation | one-period log difference, `np.diff(np.log(p))` |
| sample span | 1,000 simulated days, seed 0, no calendar attached |
| what it did **not** fix | differencing did not make returns predictable or show that real market returns have a fixed scale; that fixed scale was built into this simulation |

The last row is the important one. A transformation can solve one problem
without solving every other problem, so record what remains unknown.

</details>

<a id="real-data-companion-us-market-wealth-returns-and-tail-days"></a>

### Real-data companion — US market wealth, returns, and tail days · Deck B · optional after S4

> **Optional empirical comparison:** Deck B, after recording segment S4. Run this after the known-truth demo. Use the same transformation and compare the simulation with the historical series. Record one important difference and the most likely reason for it.

In [ ]:
market_daily_return, market_source = load_market_returns()
market_wealth = (1 + market_daily_return).cumprod()
market_recent = market_daily_return.loc["2000":]

print("Source: Kenneth R. French Data Library, bundled daily US market factor")
print(f"File: {market_source}")
print(f"Sample: {market_daily_return.index.min().date()} through {market_daily_return.index.max().date()}")
print("Units: growth of $1 for the level; decimal simple return for the change")
print(f"2000+ annualized volatility: {market_recent.std() * np.sqrt(252):.1%}")
print(f"2000+ share beyond 3 sample SD: {(market_recent.abs() > 3 * market_recent.std()).mean():.2%}")
print("Largest absolute daily moves since 2000:")
print(market_recent.loc[market_recent.abs().nlargest(5).index].sort_index().map("{:+.2%}".format))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(market_wealth.loc["2000":], lw=0.8)
axes[0].set_yscale("log")
axes[0].set_title("US market wealth index (log scale)")
axes[0].set_ylabel("Growth of $1 since 1926")
axes[1].plot(market_recent, lw=0.45, color="tab:red")
axes[1].axhline(0, color="0.3", lw=0.5)
axes[1].set_title("Daily simple returns: bounded, but not tame")
axes[1].set_ylabel("Simple return")
fig.tight_layout(); plt.show()
print("One history cannot identify a data-generating process; it can show which realized patterns need explanation.")

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| what the simulation showed | `[the known-truth result you are comparing against]` |
| what the real series showed | `[sign, scale, and any figure you read off]` |
| where they disagree | `[the specific difference]` |
| the most likely reason | `[a property of real data the simulation omitted]` |
| what you did **not** change | `[confirm the procedure stayed fixed]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**A completed comparison**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| what the simulation showed | a random walk whose differences were stable in scale; the level wandered, the returns did not |
| what the real series showed | US market wealth 1926–2026 wanders upward; daily returns sit near zero, but a handful of days are enormous |
| where they disagree | the simulated returns have no comparable tail; the worst real days are far larger than the simulation ever produces |
| the most likely reason | the simulation draws from a fixed light-tailed distribution, so it has no crashes and no volatility clustering |
| what I did **not** change | the transformation, the sample span, and the plotting procedure were held fixed |

The difference is the finding. The simulation is a controlled world whose rules
we know; the real series is what we ultimately want to understand. Explain the
gap rather than changing the simulation after seeing the historical result.

</details>

<a id="demo-4-read-memory-from-the-acf"></a>

### Demo 4 — Read memory from the ACF · Deck B · run at the break after S5

> **Break cue:** Deck B, after recording segment S5. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
T_memory = 700
memory_rng = np.random.default_rng(7057)
market_daily_return, _ = load_market_returns()
market_recent = market_daily_return.loc["2000":]

ar_shocks = memory_rng.standard_normal(T_memory)
ar_series = np.zeros(T_memory)
for t in range(1, T_memory):
    ar_series[t] = 0.72 * ar_series[t - 1] + ar_shocks[t]

ma_shocks = memory_rng.standard_normal(T_memory + 1)
ma_series = ma_shocks[1:] + 0.72 * ma_shocks[:-1]

memory_series = {
    "AR(1): past values echo": ar_series,
    "MA(1): one old shock lingers": ma_series,
    "US market return": market_recent.to_numpy(),
    "Squared US market return": market_recent.to_numpy() ** 2,
}

fig, axes = plt.subplots(2, 2, figsize=(10, 6.5), sharex=True)
for ax, (name, values) in zip(axes.flat, memory_series.items()):
    rho = acf(values, nlags=21, fft=True)
    markerline, stemlines, baseline = ax.stem(range(1, 22), rho[1:])
    plt.setp(markerline, markersize=3.5)
    plt.setp(stemlines, linewidth=1.2)
    baseline.set_color("0.65")
    bound = 1.96 / np.sqrt(len(values))
    ax.axhline(bound, color="tab:red", ls="--", lw=0.8)
    ax.axhline(-bound, color="tab:red", ls="--", lw=0.8)
    ax.set_title(name)
    ax.set_ylabel("autocorrelation")
    ax.spines[["top", "right"]].set_visible(False)
for ax in axes[-1]:
    ax.set_xlabel("lag")
fig.suptitle("Different memories leave different ACF fingerprints", fontweight="bold")
fig.tight_layout()
plt.show()

print("AR(1): the population ACF fades geometrically across lags.")
print("MA(1): the population ACF is zero after lag 1; sample bars still wobble.")
print("Market: compare weak signed-return memory with persistent magnitude memory.")


adf_rng = np.random.default_rng(0)
adf_steps = adf_rng.standard_normal(1000) * 0.01
adf_prices = 100 * np.exp(np.cumsum(adf_steps))
adf_returns = np.diff(np.log(adf_prices))
adf_log_price = adf_summary("log price", np.log(adf_prices))
adf_log_return = adf_summary("log return", adf_returns)
# Expected under this known-truth construction: fail to reject for the log
# price and reject for the log return. Neither output certifies a permanent
# description of an unknown real-world process.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| series and transformation | `[series examined and transformation applied]` |
| lag range | `[first and last lag, plus any reference band]` |
| observed memory pattern | `[specific ACF values or shape]` |
| closest classical vocabulary | `[AR, MA, ARMA, or another description]` |
| what the ACF cannot establish | `[bounded limitation]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**What a complete entry looks like** — the slide asks for *series and
transformation; lag range; observed memory pattern; closest classical
vocabulary; one claim the ACF cannot establish*.

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| series and transformation | simulated AR(1), phi = 0.72, 700 observations, seed 7057; no transformation applied |
| lag range | lags 1 through 21; rough white-noise reference band ±0.074 = 1.96/sqrt(700) |
| observed memory pattern | 0.70 at lag 1, 0.45 at lag 2, 0.28 at lag 3 — a gradual fade |
| closest classical vocabulary | AR: earlier **values** echo. The MA(1) panel is the contrast — 0.45 at lag 1, then inside the band from lag 2 on |
| what the ACF cannot establish | that the process *is* an AR(1); that the pattern persists out of sample; or that any of it is tradeable |

Compare the last two rows against the market panels in the same figure. Raw
returns show almost nothing; squared returns show a slow decay. Same series, two
different questions — direction is faint, magnitude clusters.

The small bars after lag 1 in the MA(1) panel are ordinary sample variation.
That is why an ACF suggests models to compare rather than proving the process's
exact order by itself.

</details>

<a id="demo-5-spurious-regression-on-two-independent-random-walks"></a>

### Demo 5 — Spurious regression on two independent random walks · Deck B · run at the break after S6

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
T = 500
n_trials = 200
demo5_rng = np.random.default_rng(0)
big_t_stats = 0
for _ in range(n_trials):
    a = np.cumsum(demo5_rng.standard_normal(T))
    b = np.cumsum(demo5_rng.standard_normal(T))
    # Simple OLS of b on a
    A = np.column_stack([np.ones(T), a])
    beta, *_ = np.linalg.lstsq(A, b, rcond=None)
    resid = b - A @ beta
    sigma2 = resid @ resid / (T - 2)
    se = np.sqrt(sigma2 * np.linalg.inv(A.T @ A)[1, 1])
    t = beta[1] / se
    if abs(t) > 2:
        big_t_stats += 1

print(f"Of {n_trials} independent random-walk pairs, "
      f"{big_t_stats} had |t| > 2 on the slope coefficient.")
# Expected with seed 0: 181 of 200 exceed |t| > 2, far above the classical 5%.
# The usual t-statistic reference distribution is invalid for this regression.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| object | `[series being modeled]` |
| fitted relation | `[regression or comparison performed]` |
| evidence | `[specific output that supports the claim]` |
| boundary | `[simulation, sample, and seed]` |
| limitation | `[what the demonstration does not establish]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**Completed diagnostic entry**

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| object | two independently simulated wandering price levels |
| fitted relation | an OLS level-on-level regression, included as a known-false comparison |
| evidence | 181 of 200 regressions have an absolute slope t-statistic above 2 even though the underlying random shocks are independent |
| boundary | 200 simulated pairs, 500 observations per series, seed 0; both levels accumulate shocks over the same sample length |
| limitation | this demonstrates spurious level regression, not that every regression between financial levels is false |

The appropriate next question is whether there is an economic reason to expect
some combination of the levels to be stable—not whether the first regression
looks impressive.

</details>

<a id="demo-6-a-cointegrated-pair-and-its-stationary-spread"></a>

### Demo 6 — A cointegrated pair, and its stationary spread · Deck B · run at the break after S6

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
T = 1000
demo6_rng = np.random.default_rng(0)
shared = np.cumsum(demo6_rng.standard_normal(T))          # common stochastic trend
x = shared + demo6_rng.standard_normal(T) * 0.5
y = 0.8 * shared + demo6_rng.standard_normal(T) * 0.5     # noisy linear function
spread = y - 0.8 * x

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(x, label="x", lw=0.8)
axes[0].plot(y, label="y", lw=0.8)
axes[0].set_title("Two cointegrated series (both non-stationary)")
axes[0].legend()
axes[1].plot(spread, color="tab:purple", lw=0.6)
axes[1].set_title("Their spread y − 0.8x (stationary)")
fig.tight_layout()
plt.show()

adf_summary("x", x)
adf_summary("y", y)
adf_summary("spread", spread)
# Spread should be stationary (low p); the levels are not.

stat, p, _ = coint(x, y)
print(f"Engle-Granger cointegration test: stat = {stat:+.3f}, p = {p:.4f}")
# Expected: x and y each fail to reject a unit root; their spread rejects decisively
#           and Engle-Granger rejects. Cointegration belongs to the pair, not to either series.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| series | `[levels or spread examined]` |
| relation tested | `[formula and any fitted parameter]` |
| result | `[statistic, p-value, or diagnostic]` |
| what it cannot establish | `[bounded limitation]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**What a complete entry looks like** — record what you tested, on what, and
what conclusion the result supports.

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| series | two simulated levels sharing one stochastic trend, 1,000 days, seed 0 |
| relation tested | plotted spread = y − 0.8x, using the coefficient built into the simulation; Engle–Granger separately estimates a level relation |
| result | spread ADF stat −32.81 and Engle–Granger stat −32.80, both p ≈ 0.0000; each individual level fails to reject its unit-root null |
| what it cannot establish | that a real pair is cointegrated, that β is stable, or that the spread is tradeable after costs |

The coefficient 0.8 is known only because we constructed the simulation. In a
real study, estimate the relation using information available at the decision
and evaluate its stability on later data.

</details>

<a id="demo-7-multiple-testing-teaser-connects-to-part-a-and-ch-16"></a>

### Demo 7 — Multiple-testing teaser (connects to Part A and Ch 16) · Deck B · run at the break after S6

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Fill in the **research-record entry** below the output — the fields are named for you, and a completed example sits under it once you have written yours.

In [ ]:
# Generate 1000 random strategies on independent IID noise; report the
# best-Sharpe one. This is the basis for the "deflated Sharpe" intuition
# we'll come back to in Ch 16.
T = 252 * 3  # 3 years of daily returns
n_strats = 1000
demo7_rng = np.random.default_rng(0)
strats = demo7_rng.standard_normal((n_strats, T)) * 0.01

sharpes = strats.mean(axis=1) / strats.std(axis=1) * np.sqrt(252)
print(f"True Sharpe of each strategy: 0 (IID noise, no skill).")
print(f"Mean Sharpe across strats:  {sharpes.mean():+.3f}")
print(f"Best Sharpe among {n_strats}:    {sharpes.max():+.3f}")
print(f"Worst Sharpe:                  {sharpes.min():+.3f}")
# The 'best' one is impressive — and pure luck.
# Expected with seed 0: true Sharpe is 0 for every strategy. The sample mean is
# about +0.02 and the best of 1,000 is about +1.83. That number is search, not skill.

#### Your research-record entry

*Double-click this cell to edit it, then press Shift-Enter to render.*

| FIELD | YOUR ENTRY |
|:--|:--|
| candidates compared | `[number and construction]` |
| development evidence | `[winning result and relevant comparison]` |
| later period | `[untouched period, or why none exists]` |
| claim warranted | `[narrow conclusion]` |

<details>
<summary><b>Compare with a completed entry</b> (write yours first, then click to open)</summary>

The fields below are the same fields, in the same order, as the editable cell above. Matching the wording is not required; matching the specificity is.

**What a complete entry looks like** — the slide asks for *candidates
compared; the development evidence; the later period; the claim the memo can
warrant*.

| FIELD | EXAMPLE ENTRY |
|:--|:--|
| candidates compared | 1,000 strategies on i.i.d. noise, 3 years of daily returns each, seed 0 |
| development evidence | best Sharpe +1.83; mean across all 1,000 ≈ 0.02 |
| later period | none — every series is noise by construction, so the true Sharpe is exactly 0 |
| claim warranted | **none about skill.** The +1.83 is what the maximum of 1,000 noise draws can look like |

The winning number looks impressive only until we reveal the search that
produced it. That is why the number of candidates belongs in the record.

</details>